# CEREBRO — 02A Artifact Intake

## EXP-INGEST-001 — File Inspection

### Purpose

When I upload an artifact to CEREBRO, the system should first understand what it can determine directly from the file before asking me for information or using AI.

This experiment captures deterministic file facts such as filename, type, size, checksum, encoding, and timestamps. These facts will later form the foundation of the AI-prefilled artifact form.

**Principle:** Extract facts first. Infer later. Let the user confirm.

### Setup

In [1]:
from pathlib import Path
import hashlib
import mimetypes
from datetime import datetime, timezone

repo_root = Path.cwd().parents[1]

upload_path = (
    repo_root
    / "poc/data/raw/text/benchmark_001.txt"
)

assert upload_path.exists(), (
    f"Uploaded artifact not found: {upload_path}"
)

print("✓ Artifact received")
print("File:", upload_path.name)

✓ Artifact received
File: benchmark_001.txt


### Inspect deterministic file facts

In [2]:
file_stat = upload_path.stat()

mime_type, encoding_hint = mimetypes.guess_type(
    upload_path.name
)

file_bytes = upload_path.read_bytes()

sha256 = hashlib.sha256(
    file_bytes
).hexdigest()

file_facts = {
    "filename": upload_path.name,
    "extension": upload_path.suffix.lower(),
    "mime_type": mime_type,
    "size_bytes": file_stat.st_size,
    "sha256": sha256,
    "encoding_hint": encoding_hint,
    "modified_at": datetime.fromtimestamp(
        file_stat.st_mtime,
        tz=timezone.utc
    ).isoformat()
}

file_facts

{'filename': 'benchmark_001.txt',
 'extension': '.txt',
 'mime_type': 'text/plain',
 'size_bytes': 294,
 'sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832',
 'encoding_hint': None,
 'modified_at': '2026-09-22T02:42:06.964731+00:00'}

### Detect text encoding

In [3]:
def detect_text_encoding(file_bytes):
    try:
        file_bytes.decode("utf-8")
        return {
            "encoding": "utf-8",
            "readable": True
        }

    except UnicodeDecodeError:
        return {
            "encoding": "unknown",
            "readable": False
        }


encoding_result = detect_text_encoding(file_bytes)

file_facts.update(encoding_result)

encoding_result

{'encoding': 'utf-8', 'readable': True}

### Create staged artifact

In [4]:
staged_artifact = {
    "status": "staged",

    "file": file_facts,

    "intake": {
        "inspection_method": "deterministic",
        "ai_used": False,
        "user_reviewed": False,
        "submitted": False
    }
}

staged_artifact

{'status': 'staged',
 'file': {'filename': 'benchmark_001.txt',
  'extension': '.txt',
  'mime_type': 'text/plain',
  'size_bytes': 294,
  'sha256': '7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832',
  'encoding_hint': None,
  'modified_at': '2026-09-22T02:42:06.964731+00:00',
  'encoding': 'utf-8',
  'readable': True},
 'intake': {'inspection_method': 'deterministic',
  'ai_used': False,
  'user_reviewed': False,
  'submitted': False}}

### Compare against current benchmark

In [5]:
import json

benchmark_path = (
    repo_root
    / "poc/data/ground_truth/CEREBRO-GT-v0.1.json"
)

with benchmark_path.open("r", encoding="utf-8") as f:
    benchmark = json.load(f)

expected_artifact = benchmark["artifacts"][0]

print(
    "Expected SHA-256:",
    expected_artifact["source"]["sha256"]
)

print(
    "Detected SHA-256:",
    file_facts["sha256"]
)

Expected SHA-256: 7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832
Detected SHA-256: 7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832


In [6]:
assert (
    file_facts["filename"]
    == expected_artifact["source"]["filename"]
)

assert (
    file_facts["sha256"]
    == expected_artifact["source"]["sha256"]
)

assert file_facts["readable"] is True

print("✓ Filename correctly identified")
print("✓ Source integrity correctly identified")
print("✓ Text artifact readable")

print("\nEXP-INGEST-001: PASS")

✓ Filename correctly identified
✓ Source integrity correctly identified
✓ Text artifact readable

EXP-INGEST-001: PASS


### Show what CEREBRO knows so far

flowchart LR
    U[User Upload] --> S[Staging]
    S --> F[File Inspection]
    F --> FF[File Facts]
    FF --> P[Prefill Foundation]
    P --> X[STOP]

In [7]:
print("CEREBRO Artifact Intake")
print("-----------------------")

print("File       :", file_facts["filename"])
print("Type       :", file_facts["mime_type"])
print("Extension  :", file_facts["extension"])
print("Encoding   :", file_facts["encoding"])
print("Size       :", file_facts["size_bytes"], "bytes")
print("Readable   :", file_facts["readable"])

print("\nStatus     : STAGED")
print("AI used    : NO")
print("User review: PENDING")

CEREBRO Artifact Intake
-----------------------
File       : benchmark_001.txt
Type       : text/plain
Extension  : .txt
Encoding   : utf-8
Size       : 294 bytes
Readable   : True

Status     : STAGED
AI used    : NO
User review: PENDING


### Closing Markdown

### Result

CEREBRO successfully inspected the uploaded artifact using deterministic file information only.

The artifact remains **staged** and has not yet entered the Digital Knowledge Twin.

The next experiment will inspect the artifact's contents and metadata to determine what additional fields can be prefilled without AI before introducing AI-assisted enrichment.